In [ ]:
"""Day 25 / Session 2 - Lab Assignment 1: Multi-Head Self-Attention from Scratch."""
import numpy as np


def softmax(x, axis=-1):
    x = np.asarray(x, dtype=float)
    e = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e / np.sum(e, axis=axis, keepdims=True)


def scaled_dot_product_attention(Q, K, V, mask=None):
    Q = np.asarray(Q, dtype=float)
    K = np.asarray(K, dtype=float)
    V = np.asarray(V, dtype=float)
    dk = Q.shape[-1]
    scores = Q @ np.swapaxes(K, -1, -2) / np.sqrt(dk)        # works with or without a leading head axis
    if mask is not None:
        scores = np.where(mask, scores, -1e9)                # False positions get a large negative score
    attn = softmax(scores, axis=-1)
    return attn @ V, attn


def split_heads(x, h):
    x = np.asarray(x, dtype=float)
    L, d_model = x.shape
    d_head = d_model // h
    return x.reshape(L, h, d_head).transpose(1, 0, 2)        # (h, L, d_head): columns k*d_head:(k+1)*d_head -> head k


def combine_heads(x):
    x = np.asarray(x, dtype=float)
    h, L, d_head = x.shape
    return x.transpose(1, 0, 2).reshape(L, h * d_head)


def multi_head_attention(X, Wq, Wk, Wv, Wo, h, mask=None):
    X = np.asarray(X, dtype=float)
    Q = split_heads(X @ Wq, h)
    K = split_heads(X @ Wk, h)
    V = split_heads(X @ Wv, h)
    out, _ = scaled_dot_product_attention(Q, K, V, mask)     # one attention per head, in parallel
    return combine_heads(out) @ Wo
